In [1]:
import warnings
warnings.filterwarnings('ignore')

In [2]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import lightgbm as lgbm

In [3]:
# 데이터 분할 및 전처리
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import LabelEncoder

# 특성 선택
from sklearn.feature_selection import SequentialFeatureSelector

# 하이퍼파라미터 튜닝
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import precision_recall_curve
# 모델 평가 지표
from sklearn.metrics import accuracy_score
from sklearn.metrics import precision_score
from sklearn.metrics import recall_score
from sklearn.metrics import f1_score

# 데이터 시각화
import seaborn as sns

### 1. 데이터 로드 및 탐색

#### 1.1 데이터 로드

In [22]:
df_trn = pd.read_csv('train.csv')
df_tst = pd.read_csv('test.csv')

### 2. 데이터 전처리

#### 2.1 범주형 데이터 인코딩

In [23]:
# 범주형 데이터를 라벨 인코딩
cat = ['job','marital','contact','poutcome','education','default','housing','loan','month']
label_encoder = {}
for col in cat:
    le = LabelEncoder()
    df_trn[col] = le.fit_transform(df_trn[col])
    df_tst[col] = le.transform(df_tst[col])
    label_encoder[col] = le

In [24]:
print(cat)

['job', 'marital', 'contact', 'poutcome', 'education', 'default', 'housing', 'loan', 'month']


#### 2.2 파생 변수 생성

In [25]:
# 계절을 반환하는 함수 정의
def get_season(month):
    if 3 <= month <= 5:
        return 'spring'
    elif 6 <= month <= 8:
        return 'summer'
    elif 9 <= month <= 11:
        return 'autumn'
    else:
        return 'winter'

In [37]:
# 데이터 전처리 함수 정의
def preprocess_data(df_trn, df_tst):

    le = LabelEncoder()
    
    #추가!!!
    df_trn['poutcome_success'] = (df_trn['poutcome'] == 'success').astype(int)
    df_tst['poutcome_success'] = (df_tst['poutcome'] == 'success').astype(int)

    # pdays 를 갖고 연락여부로 이진분류한 새로운 칼럼 생성
    df_trn['pdays_contacted'] = (df_trn['pdays'] != -1).astype(int)
    df_tst['pdays_contacted'] = (df_tst['pdays'] != -1).astype(int)
    
    # 흠....
    df_trn['duration_per_campaign'] = df_trn['duration'] / (df_trn['campaign'] + 1)
    df_tst['duration_per_campaign'] = df_tst['duration'] / (df_tst['campaign'] + 1)
    
    # day와 month를 통합하여 연속적인 시간 정보를 제공하기 위한 새로운 칼럼 생성
    df_trn['day_of_year'] = df_trn['day'] + (df_trn['month'] - 1) * 30
    df_tst['day_of_year'] = df_tst['day'] + (df_tst['month'] - 1) * 30
    
    # 계별 특성을 학습할 수 있도록 새로운 칼럼 생성
    df_trn['season'] = df_trn['month'].apply(get_season)
    df_tst['season'] = df_tst['month'].apply(get_season)
    # 범주형인 season을 라벨인코딩 수행
    df_trn['season'] = le.fit_transform(df_trn['season'])
    df_tst['season'] = le.transform(df_tst['season'])
    
    return df_trn, df_tst

In [38]:
# 전처리 함수 적용 후 할당
df_trn_preprocessed, df_tst_preprocessed = preprocess_data(df_trn,df_tst)

#### 2.3 Feature와 Target 분리

In [39]:
total_cols = [x for x in df_trn.columns]

id_cols = ['ID']
y_cols = ['label']
x_cols = [x for x in total_cols if x not in id_cols+y_cols]

In [40]:
# feature와 target분리
df_trn_x = df_trn_preprocessed[x_cols]
df_trn_y = df_trn_preprocessed[y_cols].values.ravel()
df_tst_x = df_tst_preprocessed[x_cols]

In [41]:
df_trn_x.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 31647 entries, 0 to 31646
Data columns (total 21 columns):
 #   Column                 Non-Null Count  Dtype  
---  ------                 --------------  -----  
 0   age                    31647 non-null  int64  
 1   job                    31647 non-null  int64  
 2   marital                31647 non-null  int64  
 3   education              31647 non-null  int64  
 4   default                31647 non-null  int64  
 5   balance                31647 non-null  int64  
 6   housing                31647 non-null  int64  
 7   loan                   31647 non-null  int64  
 8   contact                31647 non-null  int64  
 9   day                    31647 non-null  int64  
 10  month                  31647 non-null  int64  
 11  duration               31647 non-null  int64  
 12  campaign               31647 non-null  int64  
 13  pdays                  31647 non-null  int64  
 14  previous               31647 non-null  int64  
 15  po

### 3. 모델 정의 및 학습

#### 3.1 검증 데이터 분리

In [42]:
# 검증 데이터셋 분리
df_retrn_x, df_val_x, df_retrn_y, df_val_y = train_test_split(df_trn_x, df_trn_y, test_size=0.2, random_state=42, stratify=df_trn_y)

#### 3.2 초기 모델 정의 및 학습

In [43]:
# 초기 모델 정의
base_model = lgbm.LGBMClassifier(random_state=42, verbose=-1, class_weight='balanced')
#학습
base_model.fit(df_retrn_x, df_retrn_y)

LGBMClassifier(class_weight='balanced', random_state=42, verbose=-1)

#### 3.3 변수 중요도 분석

In [44]:
# Feature Importance 추출
importances = base_model.feature_importances_
features = df_retrn_x.columns

# 중요도를 데이터프레임으로 저장
importance_df = pd.DataFrame({
    'Feature': features,
    'Importance': importances
}).sort_values(by='Importance', ascending=False)

# 출력
print(importance_df)

                  Feature  Importance
19            day_of_year         588
11               duration         429
5                 balance         275
0                     age         273
18  duration_per_campaign         271
9                     day         214
13                  pdays         184
1                     job         126
8                 contact          93
10                  month          92
12               campaign          74
15               poutcome          71
3               education          67
6                 housing          63
14               previous          54
20                 season          48
2                 marital          41
7                    loan          28
4                 default           9
16       poutcome_success           0
17        pdays_contacted           0


#### 3.4 변수 선택법

In [46]:
# 변수 선택 모델 정의
sfs = SequentialFeatureSelector(
    base_model,
    n_features_to_select='auto',
    direction='forward',
    scoring='f1',
    cv=5,
    n_jobs=-1
)
# 변수 선택 모델 학습하기
sfs.fit(df_retrn_x, df_retrn_y)

/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted with feature names
  warnings.warn(
/opt/anaconda3/envs/pytorch/lib/python3.11/site-packages/sklearn/utils/validation.py:2739: UserWarning: X does not have valid feature names, but LGBMClassifier was fitted w

SequentialFeatureSelector(estimator=LGBMClassifier(class_weight='balanced',
                                                   random_state=42,
                                                   verbose=-1),
                          n_jobs=-1, scoring='f1')

In [47]:
# 선택된 변수 할당하기
selected_features = df_retrn_x.columns[sfs.get_support()].tolist()
#selected_features = ['age', 'housing', 'contact', 'day', 'duration', 'campaign', 'pdays', 'poutcome', 'duration_per_campaign', 'day_of_year']
print(f"선택된 변수: {selected_features}")

선택된 변수: ['age', 'housing', 'contact', 'day', 'duration', 'campaign', 'pdays', 'poutcome', 'duration_per_campaign', 'day_of_year']


In [48]:
# 선택된 변수로 데이터 재구성
df_retrn_x_selected = df_retrn_x[selected_features]
df_val_x_selected = df_val_x[selected_features]
df_tst_x_selected = df_tst_x[selected_features]

### 4. 모델 최적화

#### 4.1 OPTUNA

In [49]:
X_train, X_val = df_retrn_x_selected, df_val_x_selected
y_train, y_val = df_retrn_y, df_val_y

In [50]:
import optuna
from lightgbm import LGBMClassifier
from sklearn.model_selection import cross_val_score
from sklearn.metrics import f1_score
from lightgbm.callback import early_stopping, log_evaluation


In [51]:
def objective(trial):
    # 1) class_weight 설정
    #    - balanced 혹은 0~10 범위에서 실수로 샘플링
    use_balanced = trial.suggest_categorical('use_balanced', [True, False])
    if use_balanced:
        class_weight = 'balanced'
    else:
        w = trial.suggest_float('weight_1', 1.0, 10.0)
        class_weight = {0: 1.0, 1: w}

    param = {
        'class_weight': class_weight,
        'n_estimators': trial.suggest_int('n_estimators', 100, 1000),
        'learning_rate': trial.suggest_loguniform('learning_rate', 1e-3, 1e-0),
        'max_depth': trial.suggest_int('max_depth', 3, 12),
        'num_leaves': trial.suggest_int('num_leaves', 20, 200),
        'min_child_samples': trial.suggest_int('min_child_samples', 5, 200),
        'subsample': trial.suggest_uniform('subsample', 0.5, 1.0),
        'colsample_bytree': trial.suggest_uniform('colsample_bytree', 0.5, 1.0),
        'reg_alpha': trial.suggest_loguniform('reg_alpha', 1e-3, 10.0),
        'reg_lambda': trial.suggest_loguniform('reg_lambda', 1e-3, 10.0),
        'random_state': 42,
        'verbose': -1,
    }

    # 모델 학습 (early stopping 활용)
    model = LGBMClassifier(**param)
    model.fit(
    X_train, y_train,
    eval_set=[(X_val, y_val)],
    eval_metric='f1',
    callbacks=[
        early_stopping(stopping_rounds=100),   
        log_evaluation(period=0)            
    ]
)

    # 3.3 최종 평가
    preds = model.predict(X_val)
    return f1_score(y_val, preds)


In [52]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=50, timeout=600)  


[I 2025-05-28 23:49:06,074] A new study created in memory with name: no-name-f372b6a6-d2ec-43f2-9cad-f5c6d3150d59


Training until validation scores don't improve for 100 rounds


[I 2025-05-28 23:49:09,015] Trial 0 finished with value: 0.60828025477707 and parameters: {'use_balanced': True, 'n_estimators': 977, 'learning_rate': 0.12042070422749955, 'max_depth': 6, 'num_leaves': 177, 'min_child_samples': 68, 'subsample': 0.5967785023816267, 'colsample_bytree': 0.8635672399194756, 'reg_alpha': 0.09857964816759635, 'reg_lambda': 0.10755850594920464}. Best is trial 0 with value: 0.60828025477707.


Early stopping, best iteration is:
[796]	valid_0's binary_logloss: 0.279255
Training until validation scores don't improve for 100 rounds


[I 2025-05-28 23:49:09,678] Trial 1 finished with value: 0.6123627960716349 and parameters: {'use_balanced': False, 'weight_1': 2.7200452228363217, 'n_estimators': 313, 'learning_rate': 0.31472521130489556, 'max_depth': 7, 'num_leaves': 104, 'min_child_samples': 125, 'subsample': 0.8362409088842083, 'colsample_bytree': 0.812408079761552, 'reg_alpha': 0.15814662323432305, 'reg_lambda': 2.2192443140538}. Best is trial 1 with value: 0.6123627960716349.


...

Early stopping, best iteration is:
[126]	valid_0's binary_logloss: 0.294537
Training until validation scores don't improve for 100 rounds


[I 2025-05-28 23:51:03,980] Trial 48 finished with value: 0.579728857327308 and parameters: {'use_balanced': False, 'weight_1': 1.5471553188114382, 'n_estimators': 848, 'learning_rate': 0.970944869860568, 'max_depth': 7, 'num_leaves': 41, 'min_child_samples': 49, 'subsample': 0.7308743009085636, 'colsample_bytree': 0.9384286917503779, 'reg_alpha': 0.041046869556398385, 'reg_lambda': 0.008576135152227205}. Best is trial 39 with value: 0.6258026853473438.


Early stopping, best iteration is:
[8]	valid_0's binary_logloss: 0.23736
Training until validation scores don't improve for 100 rounds


[I 2025-05-28 23:51:04,687] Trial 49 finished with value: 0.6083874778499705 and parameters: {'use_balanced': False, 'weight_1': 2.4585316956616934, 'n_estimators': 774, 'learning_rate': 0.36557667979013025, 'max_depth': 7, 'num_leaves': 59, 'min_child_samples': 34, 'subsample': 0.7800415902320126, 'colsample_bytree': 0.8992894687054456, 'reg_alpha': 0.026899743142247058, 'reg_lambda': 0.0010602323226786034}. Best is trial 39 with value: 0.6258026853473438.


Early stopping, best iteration is:
[31]	valid_0's binary_logloss: 0.223519


In [70]:
best_params = study.best_trial.params
#best_params = {'n_estimators': 680, 'learning_rate': 0.18802531719522447, 'max_depth': 11, 'num_leaves': 77, 'min_child_samples': 97, 'subsample': 0.9125189892140894, 'colsample_bytree': 0.9036442293342724, 'reg_alpha': 1.7613349573900532, 'reg_lambda': 0.3367791707298721}
print("Best params:", best_params)


Best params: {'use_balanced': False, 'weight_1': 2.410381104360728, 'n_estimators': 937, 'learning_rate': 0.6125554040059905, 'max_depth': 7, 'num_leaves': 22, 'min_child_samples': 18, 'subsample': 0.7443910114615322, 'colsample_bytree': 0.814874509408695, 'reg_alpha': 0.14544047176751643, 'reg_lambda': 0.00279993942218545}


In [31]:
# Best params: {'n_estimators': 680, 'learning_rate': 0.18802531719522447, 'max_depth': 11, 'num_leaves': 77, 'min_child_samples': 97, 'subsample': 0.9125189892140894, 'colsample_bytree': 0.9036442293342724, 'reg_alpha': 1.7613349573900532, 'reg_lambda': 0.3367791707298721}
# optuna를 진행해보니 val_f1 score : 0.5569

#### optuna 시각화

In [54]:
from optuna.visualization import (
    plot_optimization_history,
    plot_param_importances,
    plot_parallel_coordinate,
    plot_slice,
    plot_contour
)

In [63]:
# 1) Trial별 F1-score 추이
fig1 = plot_optimization_history(study)
fig1.show()

# 2) Hyperparameter 중요도 분석
fig2 = plot_param_importances(study)
fig2.show()

# 3) 두 파라미터 조합별 성능 분포 (parallel coordinate)
fig3 = plot_parallel_coordinate(study, params=['learning_rate','n_estimators','max_depth'])
fig3.show()

# 4) 특정 파라미터 주위 성능 (slice plot)
fig4 = plot_slice(study, params=['learning_rate','subsample'])
fig4.show()

# 5) 2차원 성능 등고선 (contour plot)
fig5 = plot_contour(study, params=['learning_rate','num_leaves'])
fig5.show()

#### optuna 그래프 해석

* 파란 점(Objective Value): 각 Trial(실험)마다 얻은 Validation F1-score

* 빨간 선(Best Value): 지금까지 최고 성능이 유지된 값을 이어 그린 것

+ Trial 1→2에서 급격히 성능이 올라간 뒤, 대략 0.60∼0.62 사이에서 **플래토(plateau)** 에 진입

+ 이후 탐색 공간 전반에서 비슷한 성능이 반복 → 지금 범위 내에서는 큰 도약이 어려움을 의미

**learning_rate (≈0.45) 가 가장 큰 영향력을 보임**

* 그다음으로 n_estimators (≈0.19), min_child_samples (≈0.16)

* num_leaves, max_depth 중간,

* subsample, colsample_bytree, reg_alpha, reg_lambda 순으로 중요도 급감

* 색이 진할수록(F1 높을수록), 해당 Trial의 파라미터 값이 어떻게 분포하는지 선으로 이어 줌

* learning_rate ≈ 0.1 ∼ 0.2, max_depth ≈ 5 ∼ 8, n_estimators ≈ 600 ∼ 900 구간에서 짙은 선들이 몰려 있는 모습이 보임



* 왼쪽: learning_rate vs Objective

    * 0.01 미만 구간은 비교적 성능 저조

    * 0.05 ∼ 0.2 구간에서 F1이 전반적으로 높음

* 오른쪽: subsample vs Objective

    * 0.6 ∼ 0.8 사이에서 좋은 Trial이 많고,

    * 0.9 이상에서는 약간 성능이 떨어지는 경향

* learning_rate (x축) 와 num_leaves (y축) 의 조합별 성능 등고선

    * 중간 빛(≈0.60) 영역보다 진한 파란색(≈0.62) 영역이

* learning_rate ≈ 0.1–0.2 × num_leaves ≈ 80–150 구역에 몰려 있음

#### 4.2 최적의 threshold 찾기

In [73]:
#최적의 params 모델에 적용
best_model = LGBMClassifier(**best_params, random_state=42)

#train 데이터 로 학습
best_model.fit(X_train, y_train)

# 예측 확률값
y_prob = best_model.predict_proba(X_val)[:, 1]  # 양성 클래스 확률

# Precision, Recall, Threshold 계산
precisions, recalls, thresholds = precision_recall_curve(y_val, y_prob)

# F1 Score 계산
f1_scores = 2 * (precisions * recalls) / (precisions + recalls + 1e-9)

# 최적 Threshold 선택
best_threshold = thresholds[np.argmax(f1_scores)]
print(f"Best Threshold for F1 Score: {best_threshold}")

# 최적 Threshold 기반 예측
y_pred = (y_prob >= best_threshold).astype(int)

# F1 Score 확인
print(f"Optimized F1 Score: {f1_score(y_val, y_pred)}")


Best Threshold for F1 Score: 1.1559446045754727e-284
Optimized F1 Score: 0.406827880512091


In [ ]:
#Best Threshold for F1 Score: 0.1855294046582513
#Optimized F1 Score: 0.6052060737527115

### 5. 모델 평가

In [74]:
# threshold 설정
thres=0.1855

In [75]:
# 훈련 데이터에 대한 성능 평가
y_retrn_pred_proba = best_model.predict_proba(X_train)[:, 1]
y_retrn_pred_label = (y_retrn_pred_proba > thres).astype(int)

print(f"훈련 데이터 Accuracy: {accuracy_score(y_train, y_retrn_pred_label)}")
print(f"훈련 데이터 Precision: {precision_score(y_train, y_retrn_pred_label)}")
print(f"훈련 데이터 Recall: {recall_score(y_train, y_retrn_pred_label)}")
print(f"훈련 데이터 F1 Score: {f1_score(y_train, y_retrn_pred_label)}")

훈련 데이터 Accuracy: 0.8711932693447091
훈련 데이터 Precision: 0.44485429730726667
훈련 데이터 Recall: 0.4071573261309926
훈련 데이터 F1 Score: 0.4251718667371761


검증 데이터로 성능 평가

In [68]:
# 검증 데이터에 대한 성능 평가
y_val_pred_proba = best_model.predict_proba(X_val)[:, 1]
y_val_pred_label = (y_val_pred_proba > thres).astype(int)

print(f"검증 데이터 Accuracy: {accuracy_score(y_val, y_val_pred_label)}")
print(f"검증 데이터 Precision: {precision_score(y_val, y_val_pred_label)}")
print(f"검증 데이터 Recall: {recall_score(y_val, y_val_pred_label)}")
print(f"검증 데이터 F1 Score: {f1_score(y_val, y_val_pred_label)}")

검증 데이터 Accuracy: 0.8684044233807267
검증 데이터 Precision: 0.4283513097072419
검증 데이터 Recall: 0.37567567567567567
검증 데이터 F1 Score: 0.40028797696184304


### 6. 최종 모델 학습(전체 훈련 데이터)

In [58]:
X_full = pd.concat([X_train, X_val], axis=0, ignore_index=True)
y_full = np.concatenate([y_train, y_val])

In [59]:
df_trn_x_selected = X_full[selected_features]
best_model.fit(df_trn_x_selected, y_full)

LGBMClassifier(colsample_bytree=0.814874509408695,
               learning_rate=0.6125554040059905, max_depth=7,
               min_child_samples=18, n_estimators=937, num_leaves=22,
               random_state=42, reg_alpha=0.14544047176751643,
               reg_lambda=0.00279993942218545, subsample=0.7443910114615322,
               use_balanced=False, weight_1=2.410381104360728)

### 7. 테스트 데이터에 대한 예측 후 저장하기

#### 7.1 테스트데이터에 대한 예측 수행

In [60]:
y_tst_pred_proba = best_model.predict_proba(df_tst_x_selected)[:,1]

In [61]:
y_tst_pred_label = (y_tst_pred_proba > thres).astype(int)

NameError: name 'thres' is not defined

#### 7.2 예측결과 저장하기

In [ ]:
df_submission = pd.read_csv('Ho_Optuna_try01')

In [ ]:
df_submission.head()

,ID,label
0,test00001,0
1,test00002,0
2,test00003,0
3,test00004,0
4,test00005,0


In [ ]:
df_submission['label'] = y_tst_pred_label

In [ ]:
df_submission['label'].value_counts()

label
0    11541
1     2023
Name: count, dtype: int64

In [ ]:
# 다하고 눌러라!!!!
filename = 'LightGBM_Ho_final-1.csv'
df_submission.to_csv(filename, index=False)